# Diffraction Orders as Branches

A sequential trace through a grating follows one diffraction order, the
`grating_order` of the surface. `BranchTracer` traces every listed order as
its own branch. The diffractive interaction model lists the orders
(`orders`) and the fraction of the incident power that each order carries
(`efficiency`: a number per order, or a callable of the wavelength and the
incidence angles in the grating frame). The key entry of an order is
`(label, "T", "m+1")` through a transmission grating and `(label, "F", "m+1")`
at a reflective one. An order that does not propagate is booked as
`evanescent` in the power ledger, and the power of the orders that are not
listed is `absorbed`.

In [1]:
import numpy as np

from optiland.materials import IdealMaterial
from optiland.optic import Optic
from optiland.rays import PolarizationState
from optiland.raytrace.branches import BranchTracer

wavelength = 0.5  # µm


def transmission_grating(period, orders, efficiency, field_deg=0.0):
    """A plane transmission grating in air (grooves along x, so the grating
    vector is along y) and an image plane 10 mm behind it."""
    optic = Optic()
    optic.surfaces.add(index=0, radius=np.inf, thickness=np.inf)
    optic.surfaces.add(
        index=1, thickness=10.0, is_stop=True, surface_type="grating",
        grating_order=1, grating_period=period, groove_orientation_angle=0.0,
        material=IdealMaterial(1.0), comment="G",
    )  # fmt: skip
    optic.surfaces.add(index=2, material=IdealMaterial(1.0))
    optic.set_aperture(aperture_type="EPD", value=4.0)
    optic.fields.set_type(field_type="angle")
    optic.fields.add(y=0.0)
    optic.fields.add(y=field_deg or 1.0)
    optic.wavelengths.add(value=wavelength, is_primary=True)
    optic.updater.set_polarization(PolarizationState(is_polarized=False))
    model = optic.surfaces[1].interaction_model
    model.orders = orders
    model.efficiency = efficiency
    return optic

## Five orders with an efficiency table

A 2 µm grating at 0.5 µm has five propagating orders at normal incidence.
Each branch leaves at the angle of the grating equation,
sin θ_m = sin θ_in + m λ / d, and carries its efficiency. The table adds to
0.9: the missing 0.1 is booked as `absorbed`.

In [2]:
efficiency = {-2: 0.05, -1: 0.2, 0: 0.3, 1: 0.3, 2: 0.05}
grating = transmission_grating(2.0, orders=(-2, -1, 0, 1, 2), efficiency=efficiency)
result = BranchTracer(grating).trace_all(num_rays=7)

for key, branch in result.branches.items():
    order = int(key[0][2][1:])
    M = float(np.mean(np.asarray(branch.rays.M)))
    print(
        f"{key[0]}: power {branch.power:.6f}, M = {M:+.9f},"
        f" grating equation {order * wavelength / 2.0:+.9f}"
    )
print({name: round(value, 12) for name, value in result.ledger.as_dict().items()})

('G', 'T', 'm-2'): power 0.050000, M = -0.500000000, grating equation -0.500000000
('G', 'T', 'm-1'): power 0.200000, M = -0.250000000, grating equation -0.250000000
('G', 'T', 'm0'): power 0.300000, M = +0.000000000, grating equation +0.000000000
('G', 'T', 'm+1'): power 0.300000, M = +0.250000000, grating equation +0.250000000
('G', 'T', 'm+2'): power 0.050000, M = +0.500000000, grating equation +0.500000000
{'kept': 0.9, 'returned': 0.0, 'pruned': 0.0, 'unfollowed': 0.0, 'escaped': 0.0, 'evanescent': 0.0, 'clipped': 0.0, 'absorbed': 0.1}


## An order that does not propagate

With a 0.8 µm period the orders ±2 would need |sin θ| = 1.25: they are
evanescent. Their share of the power is booked as `evanescent`, and the
ledger still adds to 1.

In [3]:
fine = transmission_grating(0.8, orders=(-2, -1, 0, 1, 2), efficiency=efficiency)
with np.errstate(invalid="ignore"):  # the evanescent orders have no real direction
    result = BranchTracer(fine).trace_all(num_rays=7)
for key, branch in result.branches.items():
    print(f"{key[0]}: power {branch.power:.6f}")
ledger = result.ledger
print(
    f"evanescent {ledger.evanescent:.6f}, absorbed {ledger.absorbed:.6f},"
    f" total {ledger.total():.12f}"
)

('G', 'T', 'm-1'): power 0.200000
('G', 'T', 'm0'): power 0.300000
('G', 'T', 'm+1'): power 0.300000
evanescent 0.100000, absorbed 0.100000, total 1.000000000000


## Oblique incidence

At a field angle of 10° in the y–z plane the grating equation holds for the
tangential direction cosine: M_m = M_in + m λ / d.

In [4]:
oblique = transmission_grating(
    2.0, orders=(-1, 0, 1), efficiency={-1: 0.3, 0: 0.4, 1: 0.3}, field_deg=10.0
)
result = BranchTracer(oblique).trace_all(Hy=1.0, num_rays=1)
m_in = np.sin(np.radians(10.0))
for key, branch in result.branches.items():
    order = int(key[0][2][1:])
    M = float(np.asarray(branch.rays.M)[0])
    print(f"{key[0]}: M = {M:+.12f}, M_in + m lambda / d = {m_in + order * 0.25:+.12f}")

('G', 'T', 'm-1'): M = -0.076351822333, M_in + m lambda / d = -0.076351822333
('G', 'T', 'm0'): M = +0.173648177667, M_in + m lambda / d = +0.173648177667
('G', 'T', 'm+1'): M = +0.423648177667, M_in + m lambda / d = +0.423648177667


`BranchTracer.sequence(key)` traces one order as a `SequencedOptic`, and
the reflective case keeps the listed order of the surfaces (the key side
`"F"`), so a spectrometer with a reflective grating splits the same way.